In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01001
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.525000000000

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  60.925321626225696
Gradient descend method:  None
RUN  1 , total integrated cost =  38.423635440316694
RUN  2 , total integrated cost =  38.08480815823706
RUN  3 , total integrated cost =  37.747325602491046
RUN  4 , total integrated cost =  37.528812794880174
RUN  5 , total integrated cost =  37.291221514400455
RUN  6 , total integrated cost =  37.10895303526638
RUN  7 , total integrated cost =  36.89574339723048
RUN  8 , total integrated cost =  36.72938931034952
RUN  9 , total integrated cost =  36.528627514345736
RUN  10 , total integrated cost =  36.366055285893154
RUN  11 , total integrated cost =  36.146487095491814
RUN  12 , total integrated cost =  35.948447369520395
RUN  13 , total integrated cost =  35.65665398121099
RUN  14 , total integrated cost =  35.393659643091354
RUN  15 , total integrated cost =  34.81479892624267
RU

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  480 , total integrated cost =  356.3962911331163
Improved over  480  iterations in  113.35566351749003  seconds by  26.49808766956015  percent.
Problem in initial value trasfer:  Vmean_exc -56.62444043941683 -56.6244416510316
weight =  1429.2308848370851
set cost params:  1.0 1429.2308848370851 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.296540628952
Gradient descend method:  None
RUN  1 , total integrated cost =  5075.165466153191
RUN  2 , total integrated cost =  5065.727826614528
RUN  3 , total integrated cost =  5051.183372867451
RUN  4 , total integrated cost =  5041.844567308956
RUN  5 , total integrated cost =  5028.958219582492
RUN  6 , total integrated cost =  5015.320040159307
RUN  7 , total integrated cost =  4994.569252719049
RUN  8 , total integrated cost =  4985.445206960434
RUN  9 , total integrated cost =  4972.620159198732
RUN  10 , total integrated cost =  4962.497715001556
RUN  11 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  851 , total integrated cost =  2543.1650318196675
Improved over  851  iterations in  113.53930563479662  seconds by  50.05858336157383  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447256639118 -56.62447137372782
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  83.46730298230786
Gradient descend method:  None
RUN  1 , total integrated cost =  31.400401409016087
RUN  2 , total integrated cost =  31.290164363273114
RUN  3 , total integrated cost =  31.198730940876736
RUN  4 , total integrated cost =  31.119691795476413
RUN  5 , total integrated cost =  31.04770558552097
RUN  6 , total integrated cost =  30.988106751103455
RUN  7 , total integrated cost =  30.933049441665265
RUN  8 , total integrated cost =  30.883242697220847
RUN  9 , total integrated cost =  30.83728256291873
RUN  10 , total integrated cost =  30.79658347789384
RUN 

...

Control only changes marginally.
RUN  114 , total integrated cost =  23472.296214868085
Improved over  114  iterations in  28.90073728002608  seconds by  0.1967683719442732  percent.
Problem in initial value trasfer:  Vmean_exc -56.70064643568468 -56.7006482677288
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  83.92514623802884
Gradient descend method:  None
RUN  1 , total integrated cost =  25.327381211696025
RUN  2 , total integrated cost =  25.323380901888854
RUN  3 , total integrated cost =  25.319600398070232
RUN  4 , total integrated cost =  25.316353830541768
RUN  5 , total integrated cost =  25.312990709660074
RUN  6 , total integrated cost =  25.310152576798227
RUN  7 , total integrated cost =  25.30727999732453
RUN  8 , total integrated cost =  25.304873596193644
RUN  9 , total integrated cost =  25.301879234493484
RUN  10 , total integrated cost =  25.299370133826343
RU

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  238 , total integrated cost =  313.8180002081008
Improved over  238  iterations in  66.7473379932344  seconds by  1.3845117154239261  percent.
Problem in initial value trasfer:  Vmean_exc -56.70353950099579 -56.7035396974373
weight =  10607.075841666898
set cost params:  1.0 10607.075841666898 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33279.00133324418
Gradient descend method:  None
RUN  1 , total integrated cost =  33199.58183308666
RUN  2 , total integrated cost =  33199.27136695872
RUN  3 , total integrated cost =  33199.265249302356
RUN  4 , total integrated cost =  33199.1666949687
RUN  5 , total integrated cost =  33199.124951877326
RUN  6 , total integrated cost =  33199.12180659014
RUN  7 , total integrated cost =  33199.04147950835
RUN  8 , total integrated cost =  33199.00771677848
RUN  9 , total integrated cost =  33199.00500692711
RUN  10 , total integrated cost =  33199.00071148346
RUN  11 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  48 , total integrated cost =  33187.69023592937
Improved over  48  iterations in  12.02945084311068  seconds by  0.27438052121954115  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354061013496 -56.703540735368584


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2863.62103728137
set cost params:  1.0 2863.62103728137 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5087.625921969239
Gradient descend method:  None
RUN  1 , total integrated cost =  5083.837572989859
RUN  2 , total integrated cost =  5083.83246879997
RUN  3 , total integrated cost =  5083.831728031525
RUN  4 , total integrated cost =  5083.831131713095
RUN  5 , total integrated cost =  5083.828785311507
RUN  6 , total integra

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  5083.105604115525
Control only changes marginally.
RUN  13 , total integrated cost =  5083.105604115525
Improved over  13  iterations in  5.545460410416126  seconds by  0.08884925745414307  percent.
Problem in initial value trasfer:  Vmean_exc -56.62462153106009 -56.62461836143397
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  6225.7976630169305
set cost params:  1.0 6225.7976630169305 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12998.824965874257
Gradient descend method:  None
RUN  1 , total integrated cost =  12991.477205827434
RUN  2 , total integrated cost =  12991.477205827417
RUN  3 , total integrated cost =  12991.477205827412
RUN  4 , total integrated cost =  12991.47720582741
RUN  5 , total integrated cost =  12991.477205827408
RUN  6 , total integrated cost =  12991.477205827407


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  12991.477205827407
Control only changes marginally.
RUN  7 , total integrated cost =  12991.477205827407
Improved over  7  iterations in  4.577121647074819  seconds by  0.05652634038953863  percent.
Problem in initial value trasfer:  Vmean_exc -56.670563346175676 -56.67056566537996
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1026.8396256342567
set cost params:  1.0 1026.8396256342567 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8220.529125450284
Gradient descend method:  None
RUN  1 , total integrated cost =  8215.234628101865
RUN  2 , total integrated cost =  8214.14507547093
RUN  3 , total integrated cost =  8212.964410529798
RUN  4 , total integrated cost =  8211.886820787933
RUN  5 , total integrated cost =  8210.687578741927
RUN  6 , total integrated cost =  8209.615696911254
RUN  7 , total integrated cost =  8208.437762692165
RUN  8 , total integrated cost =  8207.376326927677
RUN  9 ,

...

RUN  2 , total integrated cost =  15126.08865292873
Control only changes marginally.
RUN  2 , total integrated cost =  15126.08865292873
Improved over  2  iterations in  1.3169658780097961  seconds by  1.4210854715202004e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.679999561631185 -56.67999713416703
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
no convergence
weight =  685.5252424120156
set cost params:  1.0 685.5252424120156 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  14526.788301716284
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  14526.788301716284
Control only changes marginally.
RUN  1 , total integrated cost =  14526.788301716284
Improved over  1  iterations in  0.5057990700006485  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67729232716165 -56.67729175005907
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 6
[[True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [False, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, False], [True, True], [False, False], [True, True], [True, False], [True, True], [True, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  15126.08865292873
Control only changes marginally.
RUN  1 , total integrated cost =  15126.08865292873
Improved over  1  iterations in  0.5594927743077278  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.679999561631185 -56.67999713416703
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 7
[[True, True], [True, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True

In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21.417165711659162
Gradient descend method:  None
RUN  1 , total integrated cost =  3.3365830746176766
RUN  2 , total integrated cost =  2.673922014656207
RUN  3 , total integrated cost =  2.346443838667211
RUN  4 , total integrated cost =  2.089388373868208
RUN  5 , total integrated cost =  1.985275363439301
RUN  6 , total integrated cost =  1.9502809185226777
RUN  7 , total integrated cost =  1.9241519947669998
RUN  8 , total integrated cost =  1.8844833846444446
RUN  9 , total integrated cost =  1.8632314792225992
RUN  10 , total integrated cost =  1.857567489200968
RUN  11 , total integrated cost =  1.8521732421585364
RUN  12 , total integrated cost =  1.8478631064629538
RUN  13 , total integrated cost =  1.843413575635296
RUN  14 , total integrated cost =  1.839203354786966
RUN  15 , total integrated cost =  1.8356793218373182
RUN 

RUN  3 , total integrated cost =  6.11612302084799
RUN  4 , total integrated cost =  6.078532021757984
RUN  5 , total integrated cost =  6.049424162853543
RUN  6 , total integrated cost =  6.0220381639697536
RUN  7 , total integrated cost =  6.000876867022096
RUN  8 , total integrated cost =  5.981278959800473
RUN  9 , total integrated cost =  5.9655285167095675
RUN  10 , total integrated cost =  5.951489269159192
RUN  11 , total integrated cost =  5.939964612538482
RUN  12 , total integrated cost =  5.929521252082185
RUN  13 , total integrated cost =  5.920874233398821
RUN  14 , total integrated cost =  5.913096283343529
RUN  15 , total integrated cost =  5.906545588212314
RUN  16 , total integrated cost =  5.900720448156982
RUN  17 , total integrated cost =  5.895660736124537
RUN  18 , total integrated cost =  5.891234382185657
RUN  19 , total integrated cost =  5.88733534872849
RUN  20 , total integrated cost =  5.883894733680183
RUN  30 , total integrated cost =  5.861904432519064


RUN  900 , total integrated cost =  0.8877766711986682
RUN  1000 , total integrated cost =  0.8877170058970298
Control only changes marginally.
RUN  1024 , total integrated cost =  0.8877144142433636
Improved over  1024  iterations in  84.51949917897582  seconds by  90.12971377150933  percent.
-------  85 0.47500000000000014 0.7250000000000004
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  24.298601212943844
Gradient descend method:  None
RUN  1 , total integrated cost =  17.84560154714564
RUN  2 , total integrated cost =  17.844258694504834
RUN  3 , total integrated cost =  17.841043639167392
RUN  4 , total integrated cost =  17.838205509922396
RUN  5 , total integrated cost =  17.82742440287495
RUN  6 , total integrated cost =  17.8176625939895
RUN  7 , total integrated cost =  17.767198369968842
RUN  8 , total integrated cost =  17.750320764732795
RUN  9 , total integrated cost =  17.747941027662044
RUN  10 , total integrated co

RUN  800 , total integrated cost =  35.545316050582336
RUN  900 , total integrated cost =  35.544884522890975
RUN  1000 , total integrated cost =  35.54415672173579
RUN  1100 , total integrated cost =  35.5431372646421
RUN  1200 , total integrated cost =  35.54257084239673
RUN  1300 , total integrated cost =  35.542434517476124
RUN  1400 , total integrated cost =  35.54178892683226
RUN  1500 , total integrated cost =  35.54087670451645
RUN  1600 , total integrated cost =  35.54072312135367
RUN  1700 , total integrated cost =  35.54054031695155
RUN  1800 , total integrated cost =  35.54001171151461
RUN  1900 , total integrated cost =  35.539569367016284
RUN  2000 , total integrated cost =  35.53938540809299
RUN  3000 , total integrated cost =  35.53845441568136
Control only changes marginally.
RUN  3092 , total integrated cost =  35.538349070400876
Improved over  3092  iterations in  219.2598484698683  seconds by  1.6914000560701083  percent.
-------  125 0.47500000000000014 0.850000000

RUN  90 , total integrated cost =  3.1433236236246698
RUN  100 , total integrated cost =  3.14297871358061
RUN  110 , total integrated cost =  3.1426334858961913
RUN  120 , total integrated cost =  3.1422695529467566
RUN  130 , total integrated cost =  3.1419263522449343
RUN  140 , total integrated cost =  3.1415968439101793
RUN  150 , total integrated cost =  3.1413449920001377
RUN  160 , total integrated cost =  3.141032929688283
RUN  170 , total integrated cost =  3.140162463444141
RUN  180 , total integrated cost =  3.139895223698516
RUN  190 , total integrated cost =  3.1398806112240547
RUN  200 , total integrated cost =  3.139855011986505
RUN  300 , total integrated cost =  3.13923611380616
RUN  400 , total integrated cost =  3.139207675490277
RUN  500 , total integrated cost =  3.1391647067267314
RUN  600 , total integrated cost =  3.139122344485389
RUN  700 , total integrated cost =  3.1391096732531567
RUN  800 , total integrated cost =  3.139092885828003
RUN  900 , total integ

In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.78980526720979
Gradient descend method:  None
RUN  1 , total integrated cost =  1.78980526720979
Control only changes marginally.
RUN  1 , total integrated cost =  1.78980526720979
Improved over  1  iterations in  0.17126327380537987  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002
no convergence
set cost params:  1.0 1.0 0.0
interpolate 

converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 13
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, Tr

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
